# Ablation Analysis

Effect of disabling problem definition / criteria-based refinement / data-grounded refinement.

**Source:** `outputs/ablation_evaluation/{evaluator}/{generator}/{case}/{combo}/evaluation.json`

In [1]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

%matplotlib inline

_cwd = Path.cwd()
ROOT = _cwd if (_cwd / "pyproject.toml").exists() else _cwd.parent
ABLATION_DIR = ROOT / "outputs" / "ablation_evaluation"
SINGLE_RUN_DIR = ROOT / "outputs" / "evaluation" / "single_run"
FIGURES_DIR = ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(context="paper", style="ticks")
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [2]:
CASE_NAMES = {
    "pl_age": "drug offences",
    "pl_medical_errors": "medical errors",
    "pl_personal_rights": "personal rights",
}


def fmt_label(name: str) -> str:
    return CASE_NAMES.get(name, name)


COMBO_RE = re.compile(r"pd(True|False)_sr(True|False)_dg(True|False)")

COMPONENT_ABBR = {
    "pd": "PD",
    "sr": "CBR",
    "dg": "DGR",
}
COMPONENT_FULL_NAMES = {
    "PD": "Problem Definition",
    "CBR": "Criteria-Based Refinement",
    "DGR": "Data-Grounded Refinement",
}
ABBR_LEGEND = "; ".join(f"{abbr} = {name}" for abbr, name in COMPONENT_FULL_NAMES.items())


def label_for_combo(combo: str) -> str:
    if combo == "Full":
        return "Full (schematize)"
    skip_pd, skip_sr, skip_dg = COMBO_RE.match(combo).groups()
    skipped = [
        COMPONENT_ABBR[key]
        for key, is_skipped in zip(("pd", "sr", "dg"), (skip_pd, skip_sr, skip_dg))
        if is_skipped == "True"
    ]
    return "No " + "/".join(skipped)


COMBO_ORDER = [
    "Full",
    "pdTrue_srFalse_dgFalse",
    "pdFalse_srTrue_dgFalse",
    "pdFalse_srFalse_dgTrue",
    "pdTrue_srTrue_dgFalse",
    "pdTrue_srFalse_dgTrue",
    "pdFalse_srTrue_dgTrue",
    "pdTrue_srTrue_dgTrue",
]
COMBO_LABELS = {combo: label_for_combo(combo) for combo in COMBO_ORDER}

## Load data

In [3]:
def coverage_of(schema: dict) -> float | None:
    cov = []
    for expert in schema["experts"]:
        t = expert["total_questions"]
        if t:
            cov.append(expert["covered_questions"] / t)
    return float(np.mean(cov)) if cov else None


records = []
for path in sorted(ABLATION_DIR.glob("**/evaluation.json")):
    combo     = path.parent.name
    case      = path.parent.parent.name
    generator = path.parent.parent.parent.name
    evaluator = path.parent.parent.parent.parent.name
    if not COMBO_RE.match(combo):
        continue
    ed = json.loads(path.read_text())
    for schema in ed["evaluations"]:
        cov = coverage_of(schema)
        if cov is not None:
            records.append({
                "evaluator":  evaluator,
                "generator":  generator,
                "case":       case,
                "combo":      combo,
                "coverage":   cov,
            })

ablation_df = pd.DataFrame(records)

full_records = []
for path in sorted(SINGLE_RUN_DIR.glob("**/evaluation.json")):
    case      = path.parent.name
    generator = path.parent.parent.name
    evaluator = path.parent.parent.parent.name
    if generator not in ablation_df["generator"].unique():
        continue
    ed = json.loads(path.read_text())
    for schema in ed["evaluations"]:
        cov = coverage_of(schema)
        if cov is not None:
            full_records.append({
                "evaluator":     evaluator,
                "generator":     generator,
                "case":          case,
                "schema_index":  schema["schema_index"],
                "coverage":      cov,
            })

full_df = pd.DataFrame(full_records)
last_idx = full_df.groupby(["evaluator", "generator", "case"])["schema_index"].transform("max")
full_df = (
    full_df[full_df["schema_index"] == last_idx]
    .assign(combo="Full")
    [["evaluator", "generator", "case", "combo", "coverage"]]
)

df = pd.concat([ablation_df, full_df], ignore_index=True)
print(
    f"{len(df)} schema evaluations · "
    f"{df['case'].nunique()} case(s) · "
    f"{df['combo'].nunique()} combo(s) · "
    f"{df['generator'].nunique()} generator model(s)"
)

24 schema evaluations · 3 case(s) · 8 combo(s) · 1 generator model(s)


## Table — coverage by ablation combo

In [4]:
def fmt_pct(x):
    return "-" if pd.isna(x) else f"{x:.1%}"


def fmt_diff(x):
    return "-" if pd.isna(x) else f"{x * 100:+.1f}pp"


combo_order = [c for c in COMBO_ORDER if c in df["combo"].unique()]

detail_mean = df.pivot_table(index="combo", columns="case", values="coverage", aggfunc="mean")
case_order  = [c for c in CASE_NAMES if c in detail_mean.columns]
detail_mean = detail_mean.loc[combo_order, case_order]

diff_mean = detail_mean.subtract(detail_mean.loc["Full"], axis=1)

detail_table = pd.DataFrame(index=combo_order, columns=case_order, dtype=object)
for combo in combo_order:
    for case in case_order:
        detail_table.loc[combo, case] = (
            fmt_pct(detail_mean.loc[combo, case]) if combo == "Full"
            else fmt_diff(diff_mean.loc[combo, case])
        )
detail_table.columns = [CASE_NAMES[c] for c in case_order]

overall_mean, overall_std = detail_mean.mean(axis=1), detail_mean.std(axis=1)
diff_overall_mean, diff_overall_std = diff_mean.mean(axis=1), diff_mean.std(axis=1)
detail_table["Overall"] = [
    f"{overall_mean[combo]:.1%} ± {overall_std[combo] * 100:.1f}pp" if combo == "Full"
    else f"{diff_overall_mean[combo] * 100:+.1f}pp ± {diff_overall_std[combo] * 100:.1f}pp"
    for combo in combo_order
]

detail_table.index = [COMBO_LABELS[c] for c in combo_order]
detail_table = detail_table.rename_axis("Ablation")

caption = (
    f"Coverage by ablation combo. Full row is absolute coverage; other rows are "
    f"Δ vs. Full, in percentage points. {ABBR_LEGEND}."
)
print(f"Caption: {caption}")
display(detail_table)


def latex_escape(s):
    return s.replace("%", "\\%").replace("±", "$\\pm$").replace("Δ", "$\\Delta$")


latex_table = detail_table.map(latex_escape)
print(latex_table.to_latex(
    column_format="l" + "c" * len(latex_table.columns),
    caption=latex_escape(caption),
    label="tab:ablation_coverage",
))

Caption: Coverage by ablation combo. Full row is absolute coverage; other rows are Δ vs. Full, in percentage points. PD = Problem Definition; CBR = Criteria-Based Refinement; DGR = Data-Grounded Refinement.


,drug offences,medical errors,personal rights,Overall
Ablation,,,,
Full (schematize),83.9%,51.5%,64.0%,66.5% ± 16.3pp
No PD,-31.5pp,+5.5pp,-16.9pp,-14.3pp ± 18.7pp
No CBR,+0.1pp,+0.2pp,+4.4pp,+1.5pp ± 2.5pp
No DGR,+0.7pp,+3.9pp,-7.4pp,-1.0pp ± 5.8pp
No PD/CBR,-36.2pp,-7.4pp,-4.3pp,-16.0pp ± 17.5pp
No PD/DGR,-29.4pp,-13.7pp,-18.8pp,-20.6pp ± 8.0pp
No CBR/DGR,-40.0pp,-6.2pp,-9.6pp,-18.6pp ± 18.6pp
No PD/CBR/DGR,-83.9pp,-33.1pp,-43.3pp,-53.4pp ± 26.9pp


\begin{table}
\caption{Coverage by ablation combo. Full row is absolute coverage; other rows are $\Delta$ vs. Full, in percentage points. PD = Problem Definition; CBR = Criteria-Based Refinement; DGR = Data-Grounded Refinement.}
\label{tab:ablation_coverage}
\begin{tabular}{lcccc}
\toprule
 & drug offences & medical errors & personal rights & Overall \\
Ablation &  &  &  &  \\
\midrule
Full (schematize) & 83.9\% & 51.5\% & 64.0\% & 66.5\% $\pm$ 16.3pp \\
No PD & -31.5pp & +5.5pp & -16.9pp & -14.3pp $\pm$ 18.7pp \\
No CBR & +0.1pp & +0.2pp & +4.4pp & +1.5pp $\pm$ 2.5pp \\
No DGR & +0.7pp & +3.9pp & -7.4pp & -1.0pp $\pm$ 5.8pp \\
No PD/CBR & -36.2pp & -7.4pp & -4.3pp & -16.0pp $\pm$ 17.5pp \\
No PD/DGR & -29.4pp & -13.7pp & -18.8pp & -20.6pp $\pm$ 8.0pp \\
No CBR/DGR & -40.0pp & -6.2pp & -9.6pp & -18.6pp $\pm$ 18.6pp \\
No PD/CBR/DGR & -83.9pp & -33.1pp & -43.3pp & -53.4pp $\pm$ 26.9pp \\
\bottomrule
\end{tabular}
\end{table}

